# 365 Probabilidades — Dia #005
## Qual a probabilidade de o seu salário atual te fazer feliz?

**Tipo:** Preditivo
**Data de publicação:** 2026-06-18
**Ferramenta:** Python
**Decisão analisada:** O próximo aumento vai me fazer mais feliz?
**Hashtag:** #365Probabilidades #Dia005

---

### 📖 A História

Durante anos acreditei que o próximo aumento resolveria.

Que quando chegasse no número certo, a sensação de suficiência viria. Que trabalhar mais, ganhar mais — e finalmente descansar.

Mas o número nunca chegava. Ou quando chegava, o suficiente já havia se movido para mais adiante.

A ciência chama isso de **hedonic treadmill** — a esteira hedônica. Você corre, o ponto de chegada se move. Você nunca para.

O que a pesquisa descobriu sobre dinheiro e felicidade vai contra tudo que o mercado de trabalho nos ensinou. E tem um número exato. Vamos encontrá-lo.

---

### 📚 O Conceito: Hedonic Treadmill

Em 1971, os psicólogos Brickman & Campbell propuseram uma ideia que mudou a forma como a ciência entende felicidade: **os seres humanos têm um ponto de referência hedônico fixo** — um nível base de felicidade para o qual sempre retornam, independente do que acontece.

Você ganha mais. Fica feliz por um tempo. Volta ao mesmo nível. Você perde algo. Fica infeliz por um tempo. Volta ao mesmo nível. O ponto de chegada se move junto com você. É uma esteira — você corre mas não avança.

**As três forças que mantêm a esteira girando:**

1. **Adaptação hedônica** — qualquer ganho perde impacto emocional com o tempo. O aumento vira o salário. O apartamento novo vira casa.
2. **Comparação social** — seu ponto de referência não é absoluto, é relativo aos outros. Quando você sobe, as pessoas ao redor também subiram. O que parecia muito vira médio.
3. **Aspirações crescentes** — quando você atinge uma meta, automaticamente define uma nova. R$10k/mês era o sonho. Quando chega lá, o sonho vira R$20k.

**O que não se adapta completamente:** pesquisas mais recentes — incluindo Killingsworth (2021) — mostraram que quatro dimensões resistem à esteira: relacionamentos próximos, propósito e significado, autonomia sobre o próprio tempo, e saúde. Essas dimensões continuam contribuindo para o bem-estar mesmo no longo prazo. Diferente de dinheiro e bens materiais, que adaptam rapidamente.

A esteira explica por que o próximo aumento nunca é suficiente. O modelo de hoje calcula onde exatamente a esteira acelera.

---

### 🧮 O Modelo

Três estudos que mudaram a forma como a ciência entende a relação entre renda e bem-estar subjetivo.

**Fontes:**
- Kahneman & Deaton, 2010 — *High income improves evaluation of life but not emotional well-being* — N=450.000
- Killingsworth, 2021 — *Experienced well-being rises with income, even above $75.000* — N=33.391
- Killingsworth, Kahneman & Mellers, 2023 — *Income and emotional well-being: A conflict resolved* — N=33.391

**⚠️ Nota sobre unidades — leia antes dos números:**
Os estudos originais reportam limiares em **dólares por ano** (USD/ano). Para tornar a história legível em reais e por mês — a forma como pensamos no salário no dia a dia — o modelo converte direto: USD/ano × taxa de câmbio ÷ 12 meses = R$/mês.

Por exemplo, o limiar de **US$ 75.000/ano** de Kahneman & Deaton (2010) se torna **R$ 35.625/mês**. É o mesmo valor, só expresso na unidade que usamos no dia a dia (salário mensal) em vez da unidade do paper original (renda anual).

**⚠️ Importante:** essa conversão usa a taxa de câmbio **nominal** (R$5,70 por US$1), não uma conversão por paridade de poder de compra (PPP). Isso significa que R$35.625/mês **não é** "o equivalente brasileiro" do limiar americano — é o valor americano original, apenas reescrito em reais pela cotação do dólar. Pelo critério de poder de compra (PPP, Banco Mundial), o mesmo nível de bem-estar provavelmente corresponderia a uma cifra bem menor em reais, já que o custo de vida nos EUA é mais alto que no Brasil. O modelo mantém o número não ajustado de propósito, para não maquiar a origem do dado — mas isso quer dizer que o valor em reais deve ser lido como referência da realidade americana, não como meta calibrada para o Brasil.


In [5]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

print('✅ Bibliotecas carregadas')

✅ Bibliotecas carregadas


In [6]:
# --- DADOS DA LITERATURA ---

# Kahneman & Deaton, 2010 — N=450.000
# Descoberta original: bem-estar emocional para de crescer em ~US$75.000/ano
# Bem-estar avaliativo (satisfação com a vida) continua crescendo acima disso
limiar_kahneman_usd_ano = 75000  # USD/ano

# Killingsworth, 2021 — N=33.391
# Revisão: bem-estar experiencial CONTINUA crescendo acima de US$75.000
# Mas a taxa de crescimento desacelera a partir daqui
limiar_killingsworth_usd_ano = 100000  # USD/ano — ponto de desaceleração significativa

# Killingsworth, Kahneman & Mellers, 2023 — N=33.391
# Reconciliação: para a maioria, felicidade cresce com renda sem plateau
# EXCETO para pessoas infelizes — para elas, o plateau existe em ~US$100.000
limiar_infelizes_usd_ano = 100000  # USD/ano — plateau para pessoas infelizes

# Proporção de pessoas infelizes na população (Gallup 2023)
p_infelizes = 0.18  # 18% se classificam como infelizes

fator_correcao = 0.80  # fator de ajuste padrão do projeto

# --- CONVERSÃO: USD/ano → BRL/mês (direto) ---
taxa_brl = 5.70  # taxa de câmbio aproximada usada no modelo

limiar_kahneman_brl_mes = (limiar_kahneman_usd_ano * taxa_brl) / 12
limiar_killingsworth_brl_mes = (limiar_killingsworth_usd_ano * taxa_brl) / 12

print('=' * 70)
print('  DADOS DA LITERATURA — RENDA E FELICIDADE')
print('=' * 70)
print(f'\n  Kahneman & Deaton, 2010 (N=450.000):')
print(f'  → Limiar emocional original (referência do estudo): US${limiar_kahneman_usd_ano:,.0f}/ano')
print(f'  → Equivalente em R$/mês (câmbio {taxa_brl})         : R${limiar_kahneman_brl_mes:,.0f}/mês')
print(f'\n  Killingsworth, 2021 (N=33.391):')
print(f'  → Ponto de desaceleração (referência do estudo)    : US${limiar_killingsworth_usd_ano:,.0f}/ano')
print(f'  → Equivalente em R$/mês (câmbio {taxa_brl})         : R${limiar_killingsworth_brl_mes:,.0f}/mês')
print(f'\n  Killingsworth et al., 2023:')
print(f'  → Para pessoas felizes  : sem plateau')
print(f'  → Para pessoas infelizes: plateau em US${limiar_infelizes_usd_ano:,.0f}/ano')
print(f'  → Proporção de infelizes: {p_infelizes*100:.0f}% (Gallup 2023)')
print('=' * 70)

  DADOS DA LITERATURA — RENDA E FELICIDADE

  Kahneman & Deaton, 2010 (N=450.000):
  → Limiar emocional original (referência do estudo): US$75,000/ano
  → Equivalente em R$/mês (câmbio 5.7)         : R$35,625/mês

  Killingsworth, 2021 (N=33.391):
  → Ponto de desaceleração (referência do estudo)    : US$100,000/ano
  → Equivalente em R$/mês (câmbio 5.7)         : R$47,500/mês

  Killingsworth et al., 2023:
  → Para pessoas felizes  : sem plateau
  → Para pessoas infelizes: plateau em US$100,000/ano
  → Proporção de infelizes: 18% (Gallup 2023)


In [7]:
# --- O MODELO ---
# Probabilidade de o salário atual te fazer feliz
# Curva de bem-estar por faixa salarial (valores em R$/mês, já convertidos)

# Faixas salariais mensais em BRL — os dois limiares (35.625 e 47.500) são
# os valores CONVERTIDOS calculados na célula anterior, não números soltos
salarios_brl = np.array([2000, 3000, 5000, 8000, 10000, 15000,
                          20000, round(limiar_kahneman_brl_mes), round(limiar_killingsworth_brl_mes),
                          70000, 100000])

def bem_estar(salario, tipo='feliz'):
    """
    Calcula bem-estar subjetivo (0-100) dado salário mensal em BRL.
    Baseado em Killingsworth et al. 2023.
    limiar e desaceleracao usam os valores já convertidos de USD/ano.
    """
    limiar = limiar_kahneman_brl_mes        # R$/mês — limiar Kahneman convertido
    desaceleracao = limiar_killingsworth_brl_mes  # R$/mês — ponto Killingsworth convertido

    if tipo == 'feliz':
        # Sem plateau — cresce logaritmicamente sem limite
        score = 40 + 25 * np.log1p(salario / 3000) / np.log1p(desaceleracao / 3000)
        score = np.clip(score, 0, 95)
    else:
        # Com plateau a partir do ponto de desaceleração
        if salario <= desaceleracao:
            score = 30 + 30 * np.log1p(salario / 3000) / np.log1p(desaceleracao / 3000)
        else:
            score = 60 + 5 * np.log1p((salario - desaceleracao) / 10000)
        score = np.clip(score, 0, 75)

    return score

bem_estar_felizes = np.array([bem_estar(s, 'feliz') for s in salarios_brl])
bem_estar_infelizes = np.array([bem_estar(s, 'infeliz') for s in salarios_brl])
bem_estar_medio = (1 - p_infelizes) * bem_estar_felizes + p_infelizes * bem_estar_infelizes

threshold = 65  # bem-estar >= 65 = satisfação sustentável
p_feliz_por_faixa = bem_estar_medio / 100

print('=' * 70)
print('  PROBABILIDADE DE FELICIDADE POR FAIXA SALARIAL')
print(f'  Baseado em Killingsworth et al. 2023 | {(1-p_infelizes)*100:.0f}% felizes / {p_infelizes*100:.0f}% infelizes')
print('=' * 70)
print(f"\n  {'Salário/mês':<15} {'Bem-estar felizes':<22} {'Bem-estar infelizes':<22} {'Médio'}")
print(f"  {'─'*65}")
for i, s in enumerate(salarios_brl):
    marker = ''
    if s == round(limiar_kahneman_brl_mes):
        marker = ' ← limiar Kahneman'
    elif s == round(limiar_killingsworth_brl_mes):
        marker = ' ← limiar Killingsworth'
    print(f'  R${s:>8,.0f}    {bem_estar_felizes[i]:>8.1f}/100           '
          f'{bem_estar_infelizes[i]:>8.1f}/100          '
          f'{bem_estar_medio[i]:>5.1f}{marker}')
print(f'\n  ← R${limiar_kahneman_brl_mes:,.0f}/mês = limiar Kahneman & Deaton (2010), convertido de US$75.000/ano')
print(f'  ← R${limiar_killingsworth_brl_mes:,.0f}/mês = ponto Killingsworth (2021), convertido de US$100.000/ano')
print('=' * 70)

  PROBABILIDADE DE FELICIDADE POR FAIXA SALARIAL
  Baseado em Killingsworth et al. 2023 | 82% felizes / 18% infelizes

  Salário/mês     Bem-estar felizes      Bem-estar infelizes    Médio
  ─────────────────────────────────────────────────────────────────
  R$   2,000        44.5/100               35.4/100           42.9
  R$   3,000        46.1/100               37.4/100           44.6
  R$   5,000        48.7/100               40.4/100           47.2
  R$   8,000        51.5/100               43.8/100           50.1
  R$  10,000        53.0/100               45.6/100           51.7
  R$  15,000        55.9/100               49.0/100           54.6
  R$  20,000        58.0/100               51.6/100           56.9
  R$  35,625        62.6/100               57.2/100           61.6 ← limiar Kahneman
  R$  47,500        65.0/100               60.0/100           64.1 ← limiar Killingsworth
  R$  70,000        68.3/100               65.9/100           67.8
  R$ 100,000        71.3/100    

In [8]:
# --- VISUALIZAÇÃO ---

# GRÁFICO 1 — Curva de bem-estar por salário
fig1, ax1 = plt.subplots(figsize=(12, 8))

salarios_cont = np.linspace(2000, 100000, 1000)
bef_cont = np.array([bem_estar(s, 'feliz') for s in salarios_cont])
bei_cont = np.array([bem_estar(s, 'infeliz') for s in salarios_cont])
bem_cont = (1 - p_infelizes) * bef_cont + p_infelizes * bei_cont

ax1.plot(salarios_cont / 1000, bef_cont, color='#2a8a82',
         linewidth=2.5, label=f'Pessoas felizes ({(1-p_infelizes)*100:.0f}%)')
ax1.plot(salarios_cont / 1000, bei_cont, color='#c0392b',
         linewidth=2.5, label=f'Pessoas infelizes ({p_infelizes*100:.0f}%)', linestyle='--')
ax1.plot(salarios_cont / 1000, bem_cont, color='#c8a84b',
         linewidth=2, label='Média ponderada', linestyle=':')

ax1.axvline(x=limiar_kahneman_brl_mes / 1000, color='#888880', linestyle='--', alpha=0.6)
ax1.axvline(x=limiar_killingsworth_brl_mes / 1000, color='#888880', linestyle='--', alpha=0.6)
ax1.axhline(y=65, color='#c8a84b', linestyle='--', alpha=0.4)

ax1.text(limiar_kahneman_brl_mes / 1000 + 0.5, 42,
         f'Kahneman\nR${limiar_kahneman_brl_mes/1000:.1f}k\n(US$75k/ano)', fontsize=8, color='#888880')
ax1.text(limiar_killingsworth_brl_mes / 1000 + 0.5, 50,
         f'Killingsworth\nR${limiar_killingsworth_brl_mes/1000:.1f}k\n(US$100k/ano)', fontsize=8, color='#888880')
ax1.text(2, 66, 'Satisfação\nsustentável (65)', fontsize=8, color='#c8a84b')

ax1.set_xlabel('Salário Mensal (R$ mil)', fontsize=12)
ax1.set_ylabel('Bem-estar Subjetivo (0–100)', fontsize=12)
ax1.set_title('Renda e Felicidade — A curva que ninguém te mostrou\nKahneman & Deaton (2010) · Killingsworth et al. (2021, 2023)',
              fontsize=14, pad=15)
ax1.legend(fontsize=10)
ax1.set_xlim(0, 105)

plt.tight_layout()
plt.savefig('dia-005-grafico-01-curva.png', dpi=150, bbox_inches='tight')
plt.close()
print('✅ Gráfico 1 salvo!')


# GRÁFICO 2 — Bem-estar por faixa salarial (barras)
fig2, ax2 = plt.subplots(figsize=(12, 8))

faixas = ['R$2k', 'R$5k', 'R$10k', 'R$20k',
          f'R${limiar_kahneman_brl_mes/1000:.0f}k', f'R${limiar_killingsworth_brl_mes/1000:.0f}k',
          'R$70k', 'R$100k']
salarios_faixas = [2000, 5000, 10000, 20000,
                    limiar_kahneman_brl_mes, limiar_killingsworth_brl_mes,
                    70000, 100000]
bem_faixas = [(1 - p_infelizes) * bem_estar(s, 'feliz') +
              p_infelizes * bem_estar(s, 'infeliz') for s in salarios_faixas]

cores_barras = ['#c0392b' if b < 60 else '#c8a84b' if b < 65 else '#2a8a82'
                for b in bem_faixas]

bars = ax2.bar(faixas, bem_faixas, color=cores_barras, alpha=0.85)
ax2.axhline(y=65, color='#c8a84b', linestyle='--', linewidth=1.5, alpha=0.8)
ax2.text(6.3, 66, 'Satisfação\nsustentável', fontsize=9, color='#c8a84b')
ax2.set_ylim(0, 80)
ax2.set_xlabel('Faixa Salarial Mensal', fontsize=12)
ax2.set_ylabel('Bem-estar Subjetivo (0–100)', fontsize=12)
ax2.set_title('Bem-estar por Faixa Salarial\n🔴 Insatisfatório · 🟡 Limiar · 🟢 Sustentável',
              fontsize=14, pad=15)

for bar, valor in zip(bars, bem_faixas):
    ax2.text(bar.get_x() + bar.get_width() / 2, valor + 0.8,
             f'{valor:.0f}', ha='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('dia-005-grafico-02-faixas.png', dpi=150, bbox_inches='tight')
plt.close()
print('✅ Gráfico 2 salvo!')


# GRÁFICO 3 — Distribuição Beta (Kahneman e Killingsworth)
n_kahneman = 450000
p_abaixo_kahneman = 0.60  # 60% abaixo do limiar de satisfação sustentável

n_killingsworth = 33391
p_plateau_infelizes = p_infelizes  # 18% com plateau (infelizes — Gallup 2023)

dist_kahneman = stats.beta(p_abaixo_kahneman * n_kahneman, (1 - p_abaixo_kahneman) * n_kahneman)
dist_killingsworth = stats.beta(p_plateau_infelizes * n_killingsworth, (1 - p_plateau_infelizes) * n_killingsworth)

ic_kahneman = dist_kahneman.interval(0.95)
ic_killingsworth = dist_killingsworth.interval(0.95)

p_abaixo_kahneman_corrigido = p_abaixo_kahneman * fator_correcao
p_plateau_corrigido = p_plateau_infelizes * fator_correcao

fig3, axes3 = plt.subplots(1, 2, figsize=(14, 7))

x1 = np.linspace(ic_kahneman[0] * 0.999, ic_kahneman[1] * 1.001, 1000)
y1 = dist_kahneman.pdf(x1)
axes3[0].plot(x1 * 100, y1, color='#c0392b', linewidth=2.5)
axes3[0].fill_between(x1 * 100, y1, alpha=0.2, color='#c0392b')
axes3[0].axvline(x=ic_kahneman[0] * 100, color='#c8a84b', linestyle='--', alpha=0.7)
axes3[0].axvline(x=ic_kahneman[1] * 100, color='#c8a84b', linestyle='--', alpha=0.7)
axes3[0].axvline(x=p_abaixo_kahneman * 100, color='#2a8a82', linewidth=2)
axes3[0].set_xlabel('Proporção abaixo do limiar (%)')
axes3[0].set_ylabel('Densidade')
axes3[0].set_title('Distribuição Beta\nKahneman & Deaton 2010 (N=450.000)', fontsize=12, pad=15)
axes3[0].text(ic_kahneman[0] * 100, max(y1) * 0.7,
              f'IC 95%\n[{ic_kahneman[0]*100:.1f}%, {ic_kahneman[1]*100:.1f}%]',
              color='#c8a84b', fontsize=9)

x2 = np.linspace(ic_killingsworth[0] * 0.95, ic_killingsworth[1] * 1.05, 1000)
y2 = dist_killingsworth.pdf(x2)
axes3[1].plot(x2 * 100, y2, color='#2a8a82', linewidth=2.5)
axes3[1].fill_between(x2 * 100, y2, alpha=0.2, color='#2a8a82')
axes3[1].axvline(x=ic_killingsworth[0] * 100, color='#c8a84b', linestyle='--', alpha=0.7)
axes3[1].axvline(x=ic_killingsworth[1] * 100, color='#c8a84b', linestyle='--', alpha=0.7)
axes3[1].axvline(x=p_plateau_infelizes * 100, color='#c0392b', linewidth=2)
axes3[1].set_xlabel('Proporção com plateau de felicidade (%)')
axes3[1].set_ylabel('Densidade')
axes3[1].set_title('Distribuição Beta\nKillingsworth et al. 2023 (N=33.391)', fontsize=12, pad=15)
axes3[1].text(ic_killingsworth[0] * 100, max(y2) * 0.7,
              f'IC 95%\n[{ic_killingsworth[0]*100:.1f}%, {ic_killingsworth[1]*100:.1f}%]',
              color='#c8a84b', fontsize=9)

plt.suptitle('365 Probabilidades — Dia #005 | Modelo Bayesiano com Distribuição Beta',
             fontsize=11, color='gray', y=1.02)
plt.tight_layout()
plt.savefig('dia-005-grafico-03-bayesiano.png', dpi=150, bbox_inches='tight')
plt.close()
print('✅ Gráfico 3 salvo!')

print()
print('=' * 65)
print('  MODELO BAYESIANO — RESULTADO COM FATOR DE CORREÇÃO')
print('=' * 65)
print(f'  Abaixo do limiar de satisfação — estimativa : {p_abaixo_kahneman*100:.0f}%')
print(f'  Abaixo do limiar de satisfação — IC 95%     : [{ic_kahneman[0]*100:.1f}%, {ic_kahneman[1]*100:.1f}%]')
print(f'  Abaixo do limiar de satisfação — corrigido  : {p_abaixo_kahneman_corrigido*100:.1f}%')
print()
print(f'  Com plateau de felicidade — estimativa      : {p_plateau_infelizes*100:.0f}%')
print(f'  Com plateau de felicidade — IC 95%          : [{ic_killingsworth[0]*100:.1f}%, {ic_killingsworth[1]*100:.1f}%]')
print(f'  Com plateau de felicidade — corrigido        : {p_plateau_corrigido*100:.1f}%')
print('=' * 65)

✅ Gráfico 1 salvo!
✅ Gráfico 2 salvo!


/var/folders/r2/9tt43p2560g52dz5c8_zp6zh0000gn/T/ipykernel_2651/421716114.py:69: UserWarning: Glyph 128308 (\N{LARGE RED CIRCLE}) missing from font(s) DejaVu Serif.
  plt.tight_layout()
/var/folders/r2/9tt43p2560g52dz5c8_zp6zh0000gn/T/ipykernel_2651/421716114.py:69: UserWarning: Glyph 128993 (\N{LARGE YELLOW CIRCLE}) missing from font(s) DejaVu Serif.
  plt.tight_layout()
/var/folders/r2/9tt43p2560g52dz5c8_zp6zh0000gn/T/ipykernel_2651/421716114.py:69: UserWarning: Glyph 128994 (\N{LARGE GREEN CIRCLE}) missing from font(s) DejaVu Serif.
  plt.tight_layout()
/var/folders/r2/9tt43p2560g52dz5c8_zp6zh0000gn/T/ipykernel_2651/421716114.py:70: UserWarning: Glyph 128308 (\N{LARGE RED CIRCLE}) missing from font(s) DejaVu Serif.
  plt.savefig('dia-005-grafico-02-faixas.png', dpi=150, bbox_inches='tight')
/var/folders/r2/9tt43p2560g52dz5c8_zp6zh0000gn/T/ipykernel_2651/421716114.py:70: UserWarning: Glyph 128993 (\N{LARGE YELLOW CIRCLE}) missing from font(s) DejaVu Serif.
  plt.savefig('dia-005-graf

✅ Gráfico 3 salvo!

  MODELO BAYESIANO — RESULTADO COM FATOR DE CORREÇÃO
  Abaixo do limiar de satisfação — estimativa : 60%
  Abaixo do limiar de satisfação — IC 95%     : [59.9%, 60.1%]
  Abaixo do limiar de satisfação — corrigido  : 48.0%

  Com plateau de felicidade — estimativa      : 18%
  Com plateau de felicidade — IC 95%          : [17.6%, 18.4%]
  Com plateau de felicidade — corrigido        : 14.4%


### 💡 O Insight

**60% dos trabalhadores estão abaixo do limiar de satisfação sustentável** — equivalente a R$35.625/mês, convertido do limiar original de US$75.000/ano de Kahneman & Deaton. Com 95% de confiança e intervalo de menos de 0,2 ponto percentual: quando N=450.000, a ciência não deixa margem para dúvida.

Mas o número mais revelador é o segundo: **18% das pessoas têm plateau de felicidade independente do salário.** Para essas pessoas, ganhar mais não resolve — o problema não é financeiro.

Para os outros 82%, a felicidade continua crescendo com a renda, mas em ritmo logarítmico: cada real adicional importa menos do que o anterior.

O que isso significa na prática:
- Abaixo de R$35.625/mês — cada aumento muda muito
- Entre R$35,6k e R$47,5k/mês — zona de transição crítica
- Acima de R$47.500/mês — retornos decrescentes para a maioria
- Para 18% — o dinheiro parou de ser a resposta há muito tempo

*Você está correndo numa esteira que nunca para — ou já passou do ponto onde dinheiro resolve?*

---

### ⚠️ Limitações do Modelo
- **Conversão por câmbio nominal, não por PPP**: os valores em R$/mês (R$35.625 e R$47.500) usam a cotação direta do dólar (5,70), não uma conversão por paridade de poder de compra. Pelo critério de poder de compra, o limiar equivalente para o Brasil seria provavelmente bem mais baixo — esses números refletem a realidade de renda americana, não uma meta calibrada para o custo de vida brasileiro
- Estudos americanos — renda e custo de vida no Brasil são diferentes, e o fator de correção (×0,80) é uma aproximação geral do projeto, não um ajuste de poder de compra
- Bem-estar subjetivo é autorrelatado — sujeito a viés de desejabilidade social
- Curva modelada logaritmicamente — simplificação de uma realidade mais complexa
- Proporção de pessoas infelizes (18%) baseada no Gallup — pode variar por contexto e período

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*